# Phase 2: Regime Detection (HMM Training)

In this notebook, we will train our **Hidden Markov Model (HMM)** to identify market regimes using the stationary features engineered in Phase 1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Ensure the project root is in the path
sys.path.append(os.path.abspath('..'))

from src.models.hmm_model import NomosHMM
from src.data.data_manager import DataManager

# Set plotting style
sns.set_theme(style="whitegrid")

## 1. Load Processed Features

In [ ]:
# Load the final processed features from Phase 1
data_path = "../data/raw/processed_features.csv"
if os.path.exists(data_path):
    df = pd.read_csv(data_path, index_col=0, parse_dates=True)
    print(f"Data loaded successfully: {df.shape[0]} rows")
    display(df.tail())
else:
    print("Features not found. Please run the ingestion notebook first.")

## 2. Train the HMM
We will select the features that provide the best signal for regime detection.

In [ ]:
# Define features to train on
features = ['NIFTY50_Ret', 'VIX_Spread', 'Corr_NIFTY50_Gold', 'Detrended_Vol']

# Initialize and Fit the HMM
nomos_hmm = NomosHMM(n_components=3, covariance_type="full")
nomos_hmm.fit(df, features)

print("HMM trained and regimes identified!")
print("State Labels:", nomos_hmm.state_labels)

## 3. Visualize Detected Regimes
Mapping the mathematical states back to the NIFTY50 price chart.

In [ ]:
# Get predicted regimes
df['Regime'] = nomos_hmm.predict(df)
df['Regime_Prob'] = nomos_hmm.predict_proba(df).max(axis=1)

# Visualization
plt.figure(figsize=(15, 8))
colors = {'Bull': 'green', 'Neutral': 'silver', 'Bear': 'red'}

for label, color in colors.items():
    mask = df['Regime'] == label
    plt.scatter(df.index[mask], df.loc[mask, 'NIFTY50_Ret'].cumsum(), 
                label=label, c=color, s=5, alpha=0.6)

plt.title("NIFTY50 Cumulative Returns overlayed with HMM Regimes")
plt.legend()
plt.show()

## 4. Stability Check: Transition Matrix
How likely is the market to stay in its current regime?

In [ ]:
labels = [nomos_hmm.state_labels[i] for i in range(3)]
trans_mat = pd.DataFrame(nomos_hmm.model.transmat_, index=labels, columns=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(trans_mat, annot=True, cmap="YlGnBu")
plt.title("Regime Transition Probability Matrix")
plt.show()

## 5. Save the Model

In [ ]:
nomos_hmm.save_model("../src/models/trained_hmm_v1.joblib")